# Network Health Telemetry: Visualizing DNS, TLS, and Load-Balancer Behavior

> L3 concept integration — this notebook combines **networking fundamentals** with **monitoring and observability** to visualize the health of each network layer a request traverses. Instead of probing each layer in isolation, the approach here treats DNS resolution, the TLS handshake, and load-balanced routing as a single observable pipeline: measure latency and failure at each stage, then render a combined health dashboard. This pattern is what tools like Prometheus blackbox_exporter or Datadog Network Monitoring do under the hood — the notebook is a from-scratch walkthrough of the same idea.

## Setup

The notebook uses only the Python standard library plus `matplotlib` for rendering. If matplotlib is unavailable, every visualization falls back to a text-based table so the logic stays verifiable in any environment.

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
import socket
import ssl
import time
import json
from collections import Counter, defaultdict
from urllib.request import urlopen
from urllib.error import URLError

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

## Layer 1 — DNS resolution as a measurable event

DNS is often treated as someone else's problem until it isn't. In an observability pipeline, the first thing to capture is *how long resolution takes* and *whether it returns the records you expect*. The function below resolves a hostname and returns every address the resolver offers, along with the wall-clock time the call consumed. Repeating this over time reveals resolver cache behavior, TTL expiry, and intermittent resolution failures.

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
def probe_dns(hostname, port=443, iterations=5):
    """Resolve a hostname several times; return per-iteration latency and records."""
    samples = []
    for _ in range(iterations):
        start = time.perf_counter()
        try:
            records = socket.getaddrinfo(hostname, port, socket.AF_UNSPEC, socket.SOCK_STREAM)
            ips = list({r[4][0] for r in records})
            latency_ms = (time.perf_counter() - start) * 1000
            samples.append({"ok": True, "latency_ms": latency_ms, "ips": ips})
        except socket.gaierror as exc:
            latency_ms = (time.perf_counter() - start) * 1000
            samples.append({"ok": False, "latency_ms": latency_ms, "error": str(exc)})
    return samples

dns_target = "example.com"
dns_samples = probe_dns(dns_target, iterations=6)

ok_count = sum(1 for s in dns_samples if s["ok"])
avg_latency = sum(s["latency_ms"] for s in dns_samples) / len(dns_samples)
print(f"DNS probe -> {dns_target}: {ok_count}/{len(dns_samples)} ok, avg {avg_latency:.1f} ms")
for i, s in enumerate(dns_samples, 1):
    status = "ok" if s["ok"] else f"FAIL ({s['error']})"
    print(f"  attempt {i}: {s['latency_ms']:.1f} ms  [{status}]")

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
# Visualize DNS latency across attempts to spot cache hits vs. misses
attempts = list(range(1, len(dns_samples) + 1))
latencies = [s["latency_ms"] for s in dns_samples]
colors = ["#4CAF50" if s["ok"] else "#F44336" for s in dns_samples]

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(attempts, latencies, color=colors)
    ax.set_xlabel("Probe attempt")
    ax.set_ylabel("Resolution time (ms)")
    ax.set_title(f"DNS resolution latency — {dns_target}")
    ax.axhline(avg_latency, color="#2196F3", linestyle="--", label=f"avg {avg_latency:.1f} ms")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"\nDNS latency per attempt (avg {avg_latency:.1f} ms):")
    for i, s in enumerate(dns_samples, 1):
        bar = "#" * int(s["latency_ms"] / 5 + 1)
        print(f"  attempt {i}: {bar} {s['latency_ms']:.1f} ms")

### What to look for
- The first probe is often slower (cold cache); subsequent ones should be near-zero if the resolver caches. A flat high line suggests the resolver isn't caching or the TTL is very short.
- A red bar (failure) on any attempt is a DNS-layer incident — in a real pipeline this is the signal that triggers a `dns_resolution_failed` alert before any TLS or HTTP probe even runs.

## Layer 2 — TLS handshake timing and certificate health

Once DNS returns an IP, the client opens a TCP connection and performs the TLS handshake. Two things matter observability-wise: *how long the handshake takes* (a proxy for network round-trip plus server crypto overhead) and *how close the certificate is to expiry* (a countdown that should page someone before it hits zero). The function below measures both.

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
from datetime import datetime, timezone

def probe_tls(hostname, port=443):
    """Time a TLS handshake and report certificate expiry + cipher details."""
    context = ssl.create_default_context()
    start = time.perf_counter()
    try:
        with socket.create_connection((hostname, port), timeout=10) as raw:
            with context.wrap_socket(raw, server_hostname=hostname) as tls:
                cert = tls.getpeercert()
                handshake_ms = (time.perf_counter() - start) * 1000
                cipher = tls.cipher()
                version = tls.version()
    except (socket.timeout, ssl.SSLError, OSError) as exc:
        return {"ok": False, "error": str(exc)}

    not_after = cert.get("notAfter", "")
    days_left = None
    if not_after:
        expiry = datetime.strptime(not_after, "%b %d %H:%M:%S %Y %Z").replace(tzinfo=timezone.utc)
        days_left = (expiry - datetime.now(timezone.utc)).days

    return {
        "ok": True,
        "handshake_ms": handshake_ms,
        "tls_version": version,
        "cipher": cipher[0] if cipher else None,
        "cert_not_after": not_after,
        "cert_days_left": days_left,
        "san": [name for _, name in cert.get("subjectAltName", [])],
    }

tls_target = "example.com"
tls_result = probe_tls(tls_target)

if tls_result["ok"]:
    print(f"TLS probe -> {tls_target}")
    print(f"  handshake: {tls_result['handshake_ms']:.1f} ms")
    print(f"  version:   {tls_result['tls_version']}")
    print(f"  cipher:    {tls_result['cipher']}")
    print(f"  cert expires in {tls_result['cert_days_left']} days ({tls_result['cert_not_after']})")
else:
    print(f"TLS probe -> {tls_target}: FAIL ({tls_result['error']})")

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
# Compare TLS handshake latency across several hosts on the same port
tls_hosts = ["example.com", "www.example.com", "nonexistent.invalid.tld"]
tls_results = []
for host in tls_hosts:
    r = probe_tls(host)
    tls_results.append((host, r))
    status = f"{r['handshake_ms']:.1f} ms" if r["ok"] else f"FAIL: {r['error'][:40]}"
    print(f"  {host:30} {status}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    names = [h for h, _ in tls_results]
    values = [r["handshake_ms"] if r["ok"] else 0 for _, r in tls_results]
    colors = ["#4CAF50" if r["ok"] else "#F44336" for _, r in tls_results]
    ax.bar(names, values, color=colors)
    ax.set_ylabel("TLS handshake time (ms)")
    ax.set_title("TLS handshake latency across targets")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

### What to look for
- Handshake latency is dominated by network round-trip. A sudden jump often means a routing change or an overloaded server, not a TLS problem itself.
- `cert_days_left` is the field that feeds expiry alerts. A common threshold is 30 days — enough time to renew before browsers start warning users.
- A failure on `nonexistent.invalid.tld` is expected; in a real pipeline you'd probe only known-good hosts and treat any failure as an incident.

## Layer 3 — Load-balanced routing and backend distribution

A load balancer spreads requests across a pool of backends. From the outside, you can't see the balancer's internal state, but you can infer it: send many requests from the same client and watch whether the answers change. The simulation below models three backends behind a round-robin balancer, then introduces a failure to show how the distribution shifts — the same pattern a real health-check probe would detect.

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
class SimulatedBalancer:
    """Round-robin balancer over a pool of backends, with health tracking."""
    def __init__(self, backends):
        self.backends = backends
        self._index = 0
        self.unhealthy = set()
        self.history = []

    def healthy_backends(self):
        return [b for b in self.backends if b not in self.unhealthy]
    
    def mark_unhealthy(self, name):
        self.unhealthy.add(name)

    def route(self):
        pool = self.healthy_backends()
        if not pool:
            raise RuntimeError("no healthy backends")
        backend = pool[self._index % len(pool)]
        self._index += 1
        self.history.append(backend)
        return backend

    def distribution(self):
        return dict(Counter(self.history))

pool = ["backend-a:8080", "backend-b:8080", "backend-c:8080"]
balancer = SimulatedBalancer(pool)

print("Phase 1 — all backends healthy, 9 requests:")
for i in range(9):
    print(f"  request {i+1:>2} -> {balancer.route()}")
print(f"  distribution: {balancer.distribution()}")

balancer.mark_unhealthy("backend-b:8080")
print("\nPhase 2 — backend-b marked unhealthy, 8 more requests:")
for i in range(8):
    print(f"  request {i+10:>2} -> {balancer.route()}")
print(f"  distribution: {balancer.distribution()}")

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
# Visualize how the request distribution shifts when a backend fails
dist = balancer.distribution()
names = list(dist.keys())
values = list(dist.values())
colors = ["#F44336" if n in balancer.unhealthy else "#4CAF50" for n in names]

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(names, values, color=colors)
    ax.set_ylabel("Request count")
    ax.set_title("Load-balancer distribution — backend-b failed mid-stream")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.2, str(val), ha="center")
    plt.tight_layout()
    plt.show()
else:
    print("\nRequest distribution:")
    for n in names:
        tag = " (UNHEALTHY)" if n in balancer.unhealthy else ""
        print(f"  {n}: {'#' * dist[n]} ({dist[n]}){tag}")

### What to look for
- In phase 1 the spread is even — each backend gets 3 requests. That's the healthy baseline.
- After `backend-b` is marked unhealthy, the balancer redistributes its share across the remaining two. The distribution shifts from 3/3/3 to 5/0/4 (the off-by-one is where the round-robin counter was when the failure happened).
- In a real system, the unhealthy backend would be detected by active health checks (HTTP probes from the balancer) and removed from the pool automatically. The visualization above is what that looks like from the client's perspective.

## Combined health dashboard

The three layers above are usually monitored independently — DNS in one dashboard, TLS certs in another, load-balancer pool health in a third. The integration step is to correlate them: a request that fails could be a DNS miss, a cert expiry, or a drained backend, and you want to know which layer dropped it without three separate investigations. The cell below renders a single summary across all three layers.

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
dns_ok = sum(1 for s in dns_samples if s["ok"]) / len(dns_samples)
dns_avg = sum(s["latency_ms"] for s in dns_samples) / len(dns_samples)
tls_ok = 1.0 if tls_result["ok"] else 0.0
lb_ok = len(balancer.healthy_backends()) / len(balancer.backends)

dashboard = {
    "dns": {
        "target": dns_target,
        "availability": dns_ok,
        "avg_latency_ms": round(dns_avg, 1),
        "status": "healthy" if dns_ok == 1.0 else "degraded",
    },
    "tls": {
        "target": tls_target,
        "availability": tls_ok,
        "handshake_ms": round(tls_result.get("handshake_ms", 0), 1),
        "cert_days_left": tls_result.get("cert_days_left"),
        "status": "healthy" if tls_ok else "failed",
    },
    "load_balancer": {
        "backends_total": len(balancer.backends),
        "backends_healthy": len(balancer.healthy_backends()),
        "availability": lb_ok,
        "status": "healthy" if lb_ok == 1.0 else "degraded",
    },
}

print(json.dumps(dashboard, indent=2))

In [ ]:
# last_verified: 2026-09-02 · networking concepts n/a
# Render the three-layer availability as a single dashboard strip
layers = ["DNS", "TLS", "Load Balancer"]
availability = [dashboard["dns"]["availability"], dashboard["tls"]["availability"], dashboard["load_balancer"]["availability"]]
colors = ["#4CAF50" if a == 1.0 else "#FF9800" if a >= 0.5 else "#F44336" for a in availability]

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.barh(layers, availability, color=colors)
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Availability (fraction of probes that succeeded)")
    ax.set_title("Network health dashboard — three-layer view")
    for i, a in enumerate(availability):
        ax.text(a + 0.02, i, f"{a:.0%}", va="center")
    plt.tight_layout()
    plt.show()
else:
    print("\nNetwork health dashboard:")
    for layer, a in zip(layers, availability):
        bar = "#" * int(a * 20)
        print(f"  {layer:15} {bar} {a:.0%}")

## How the layers connect

| Layer | What it measures | Failure mode | Observable signal |
|---|---|---|---|
| DNS | Name → IP resolution time and success | Resolver down, expired record, NXDOMAIN | Latency spike or `gaierror` |
| TLS | Handshake time, cert validity, cipher | Expired cert, protocol mismatch, timeout | `SSLError`, `cert_days_left < threshold` |
| Load balancer | Backend pool health and distribution | Backend failure, pool drain | Skewed distribution, 5xx rate increase |

A request that times out could be any of the three. The dashboard above lets you rule layers in or out in order: if DNS availability is 100% but TLS is failing, the problem is at the TLS layer or below — you don't need to debug the resolver. This ordered elimination is the practical value of treating the three layers as one observable pipeline rather than three independent checks.

## Where this leads

- Replace the simulated balancer with real HTTP probes against a live service and plot response-code distribution over time — that's the jump from simulation to production monitoring.
- Feed the per-layer metrics into a time-series store (Prometheus, InfluxDB) and set availability thresholds that page on sustained degradation rather than single failures.
- Add a fourth layer — HTTP response latency and status codes — to close the gap between "the network is up" and "the application is responding correctly."